# Amazon Video Games Recommender System

**Matrix Factorization & Neural Collaborative Filtering for Rating Prediction**

This notebook builds and compares two recommender system architectures — Matrix Factorization (MF) and Neural Collaborative Filtering (NCF) — on the Amazon Video Games review dataset (~500,000 user-product interactions across 400,000+ users and 63,000 items). The pipeline covers data processing, a temporal train-test split, cold-start problem analysis, model development and training, evaluation, and a segmentation-based deep dive into model performance across user activity and item popularity tiers.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, Model
import json

# Define the path to the data file (Please change this path to where you saved the data)
data_path = './Video_Games.jsonl'
# Read the data into a pandas DataFrame
my_own_data = pd.read_json(data_path, lines=True)

if len(my_own_data) > 500000:
    my_own_data = my_own_data.sample(n=500000, random_state=42).reset_index(drop=True)

# Sanity check to make sure the dataset satisfies all criteria listed at the top.

# Print the size of the dataset.
print(f"Dataset size: {len(my_own_data)} rows")

# Calculate and print the number of unique users and items.
n_users = my_own_data['user_id'].nunique()
n_items = my_own_data['parent_asin'].nunique()
print(f"Unique users: {n_users:,}")
print(f"Unique items: {n_items:,}")

# Print the first few rows of the dataset. What columns do you have?
print("\nFirst few rows:")
print(my_own_data.head())
print(f"\nColumns: {list(my_own_data.columns)}")

Dataset size: 500000 rows
Unique users: 441,318
Unique items: 63,063

First few rows:
   rating                                              title  \
0       5                        What can I say?  ZELDA=Good   
1       5                            2K brings the heat back   
2       5  Awesome kit that requires some patience to ass...   
3       5          Beautiful Graphics and Addicting Gameplay   
4       3  There's something not right about this install...   

                                                text images        asin  \
0  The origional LOZ was and still is my favorite...     []  B00006LELB   
1  2K has brought it back with this edition. I ha...     []  B07DM3LYVV   
2  It is very important to note that this device ...     []  B072V9ZBSK   
3  Let me be the first to say that this is not a ...     []  B002Q21X7Y   
4  I know what it is now, IT'S ALMOST IMPOSSIBLE ...     []  B000066TS2   

  parent_asin                       user_id               timestamp  \
0  B001

#### **Memo**

Subject: Recommendation System Strategy for Amazon Video Games Dataset

Date:    May 2026

The provided dataset covers 500,000 user-prodect interactions across over 400,00 users and 63,000 video games, including attribudes: ratings (1–5), timestamps, verified purchase flags, and helpful votes.

Base on the dataset, which is suited for preference-based strategy, we recommend implementing two models: Matrix Factorization (MF) as a baseline and Neural Collaborative Filtering (NCF) as the primary production model.

The timestamps column allows time-aware splitting for realistic model evaluation, and verified_purchase and helpful_vote could be used to weight interactions by quality in future iterations. 
These approaches could provide meaningful personalization, such as surfacing niche titles aligned with each gamer's taste, increasing cross-sell opportunities, and improving user retention through more relevant recommendations.

In [8]:
my_own_data1 = my_own_data.copy()

In [9]:
# Data processing

# keep only the columns we need for recommendation
my_own_data1 = my_own_data[['user_id', 'parent_asin', 'rating', 'timestamp']]
# rename columns for clearity
my_own_data1 = my_own_data1.rename(columns={'parent_asin': 'item_id'})
# drop rows with missing values
my_own_data1 = my_own_data1.dropna(subset=['user_id', 'item_id', 'rating'])
# make sure rating columns is in float format
my_own_data1['rating'] = my_own_data1['rating'].astype(float)
# convert timestamp to datetime and sort by timestamp
my_own_data1['timestamp'] = pd.to_datetime(my_own_data1['timestamp'])
my_own_data1 = my_own_data1.sort_values('timestamp').reset_index(drop=True)
# encode user_id and item_id to continuous integers
my_own_data1['user_idx'] = my_own_data1['user_id'].astype('category').cat.codes
my_own_data1['item_idx'] = my_own_data1['item_id'].astype('category').cat.codes

n_users = my_own_data1['user_idx'].nunique()
n_items = my_own_data1['item_idx'].nunique()


# Print the head of the processed data
print("\nProcessed data head:")
print(my_own_data1.head())


Processed data head:
                        user_id     item_id  rating           timestamp  \
0  AGSD5NPZSBL4RZBEHWQGHMJR7VEA  B0000064PW     3.0 1998-12-01 18:36:33   
1  AGOPLXUIOTBNZH6MP6YIBF3K6G6A  B00000DFE7     5.0 1999-08-12 02:26:14   
2  AGE66IMMH3GNCI6DN4N2E7LBBB7A  0345339258     4.0 1999-08-30 18:21:49   
3  AHKDYZ633W34U322UF2JAKDPQKDQ  B000031KJW     5.0 1999-11-11 05:48:26   
4  AG5WHQBKUD6I4OSBDW2BITYMKX2Q  B00001SHN8     5.0 1999-11-14 00:58:04   

   user_idx  item_idx  
0    304000       243  
1    291793       271  
2    255370        10  
3    386833      1346  
4    233882       563  


### **Train-Test Split**

In [10]:
# Please split the data into a training set and a test set
split_idx = int(len(my_own_data) * 0.8)
train_data = my_own_data1.iloc[:split_idx].reset_index(drop=True)
test_data = my_own_data1.iloc[split_idx:].reset_index(drop=True)

# Print sample training and test data      
print("Training data sample:")
print(train_data.head())
print("\nTest data sample:")
print(test_data.head())

# Print dataset shapes and make sure it is correct
print(f"\nTraining set shape: {train_data.shape}")
print(f"Test set shape:     {test_data.shape}")
print(f"\nTraining period: {train_data['timestamp'].min()} ~ {train_data['timestamp'].max()}")
print(f"Test period:     {test_data['timestamp'].min()} ~ {test_data['timestamp'].max()}")

Training data sample:
                        user_id     item_id  rating           timestamp  \
0  AGSD5NPZSBL4RZBEHWQGHMJR7VEA  B0000064PW     3.0 1998-12-01 18:36:33   
1  AGOPLXUIOTBNZH6MP6YIBF3K6G6A  B00000DFE7     5.0 1999-08-12 02:26:14   
2  AGE66IMMH3GNCI6DN4N2E7LBBB7A  0345339258     4.0 1999-08-30 18:21:49   
3  AHKDYZ633W34U322UF2JAKDPQKDQ  B000031KJW     5.0 1999-11-11 05:48:26   
4  AG5WHQBKUD6I4OSBDW2BITYMKX2Q  B00001SHN8     5.0 1999-11-14 00:58:04   

   user_idx  item_idx  
0    304000       243  
1    291793       271  
2    255370        10  
3    386833      1346  
4    233882       563  

Test data sample:
                        user_id     item_id  rating               timestamp  \
0  AF53NOS7LR5I7NIWH6LZBRUEHCWA  B09Q6GFZGB     5.0 2021-03-14 07:05:19.610   
1  AGWDTFPEGWBXJGQ4CBVSXURE566A  B0019JZBBE     1.0 2021-03-14 07:06:43.557   
2  AFPUZLALW6XCVFMVHDSYED2P4DQQ  B00FLLFGO8     5.0 2021-03-14 07:07:21.905   
3  AGEFQIBXW6OYYWFFNUUR2VYVCXNA  B071GPJPH4     

### **Cold-Start Problem**

In [12]:
# Check how many users in the test set are not in the training set
train_users = set(train_data['user_id'].unique())
test_users = set(test_data['user_id'].unique())
new_users = test_users - train_users
print(f"Users in test but not in training: {len(new_users):,}")

# Check how many items in the test set are not in the training set
train_items = set(train_data['item_id'].unique())
test_items = set(test_data['item_id'].unique())
new_items = test_items - train_items
print(f"Items in test but not in training: {len(new_items):,}")

# Check how many instances in the test set that contains a user or an item that is not in the training set
cold_start_mask = (test_data['user_id'].isin(new_users)) | (test_data['item_id'].isin(new_items))
n_cold_start = cold_start_mask.sum()
print(f"Test instances with cold-start user or item: {n_cold_start:,}")
print(f"Percentage: {n_cold_start / len(test_data) * 100:.1f}%")

Users in test but not in training: 88,957
Items in test but not in training: 13,713
Test instances with cold-start user or item: 96,780
Percentage: 96.8%


In [ ]:
# Matrix Factorization (MF)
class MatrixFactorization(Model):
    def __init__(self, n_users, n_items, embedding_dim=32):
        super(MatrixFactorization, self).__init__()
        self.user_embedding = layers.Embedding(n_users, embedding_dim,
                                               embeddings_initializer='he_normal',
                                               embeddings_regularizer=tf.keras.regularizers.l2(1e-6))
        self.item_embedding = layers.Embedding(n_items, embedding_dim,
                                               embeddings_initializer='he_normal',
                                               embeddings_regularizer=tf.keras.regularizers.l2(1e-6))
        self.user_bias = layers.Embedding(n_users, 1)
        self.item_bias = layers.Embedding(n_items, 1)

    def call(self, inputs):
        user_idx, item_idx = inputs
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(item_idx)
        user_b = tf.squeeze(self.user_bias(user_idx), axis=-1)
        item_b = tf.squeeze(self.item_bias(item_idx), axis=-1)
        dot = tf.reduce_sum(user_vec * item_vec, axis=-1)
        return dot + user_b + item_b

# Neural Collaborative Filtering (NCF)
class NCF(Model):
    def __init__(self, n_users, n_items, embedding_dim=32):
        super(NCF, self).__init__()
        self.user_embedding = layers.Embedding(n_users, embedding_dim,
                                               embeddings_initializer='he_normal')
        self.item_embedding = layers.Embedding(n_items, embedding_dim,
                                               embeddings_initializer='he_normal')
        self.fc1 = layers.Dense(64, activation='relu')
        self.fc2 = layers.Dense(32, activation='relu')
        self.output_layer = layers.Dense(1)

    def call(self, inputs):
        user_idx, item_idx = inputs
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(item_idx)
        x = tf.concat([user_vec, item_vec], axis=-1)
        x = self.fc1(x)
        x = self.fc2(x)
        return tf.squeeze(self.output_layer(x), axis=-1)

# initialize models
n_users = my_own_data1['user_idx'].nunique()
n_items = my_own_data1['item_idx'].nunique()

mf_model = MatrixFactorization(n_users, n_items, embedding_dim=32)
ncf_model = NCF(n_users, n_items, embedding_dim=32)

print(f"MF model ready | n_users: {n_users:,}, n_items: {n_items:,}")
print(f"NCF model ready | n_users: {n_users:,}, n_items: {n_items:,}")

MF model ready | n_users: 441,318, n_items: 63,063
NCF model ready | n_users: 441,318, n_items: 63,063


2026-05-02 20:45:44.928309: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-05-02 20:45:44.928713: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-05-02 20:45:44.928875: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-05-02 20:45:44.929049: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-02 20:45:44.929481: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [ ]:
# prepare train and test data for model
train_users_idx = train_data['user_idx'].values
train_items_idx = train_data['item_idx'].values
train_ratings = train_data['rating'].values.astype(np.float32)

test_users_idx = test_data['user_idx'].values
test_items_idx = test_data['item_idx'].values
test_ratings = test_data['rating'].values.astype(np.float32)

# hyperparameters
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.MeanSquaredError()

mf_model.compile(optimizer=optimizer, loss=loss_fn)
ncf_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=loss_fn)

# training MF
print("Training Matrix Factorization...")
mf_history = mf_model.fit(
    x=(train_users_idx, train_items_idx),
    y=train_ratings,
    batch_size=2048,
    epochs=5,
    validation_data=((test_users_idx, test_items_idx), test_ratings),
    verbose=1
)

# training NCF
print("\nTraining NCF...")
ncf_history = ncf_model.fit(
    x=(train_users_idx, train_items_idx),
    y=train_ratings,
    batch_size=2048,
    epochs=5,
    validation_data=((test_users_idx, test_items_idx), test_ratings),
    verbose=1
)

Training Matrix Factorization...
Epoch 1/5


2026-05-02 20:49:12.741594: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


196/196 ━━━━━━━━━━━━━━━━━━━━ 14s 62ms/step - loss: 18.2105 - val_loss: 17.1703
Epoch 2/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 59ms/step - loss: 17.1588 - val_loss: 16.7981
Epoch 3/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 59ms/step - loss: 15.5756 - val_loss: 16.4741
Epoch 4/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 59ms/step - loss: 13.2959 - val_loss: 16.2087
Epoch 5/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 59ms/step - loss: 10.7401 - val_loss: 16.0040

Training NCF...
Epoch 1/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 57ms/step - loss: 5.0613 - val_loss: 2.6700
Epoch 2/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - loss: 0.9799 - val_loss: 2.7971
Epoch 3/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - loss: 0.4310 - val_loss: 2.6581
Epoch 4/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 57ms/step - loss: 1.0128 - val_loss: 3.1891
Epoch 5/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 57ms/step - loss: 2.4530 - val_loss: 2.5297


In [ ]:
# Generate 5 predicted values and their corresponding (user,item) pairs
sample = test_data.sample(5, random_state=42).reset_index(drop=True)
sample_users = sample['user_idx'].values
sample_items = sample['item_idx'].values

mf_preds_sample = mf_model.predict((sample_users, sample_items), verbose=0)
ncf_preds_sample = ncf_model.predict((sample_users, sample_items), verbose=0)

print("Sample predictions:")
print(f"{'user_idx':<12} {'item_idx':<12} {'actual':<10} {'MF pred':<12} {'NCF pred':<12}")
for i in range(5):
    print(f"{sample_users[i]:<12} {sample_items[i]:<12} {sample['rating'].values[i]:<10} {mf_preds_sample[i]:<12.4f} {ncf_preds_sample[i]:<12.4f}")

# Report RMSE and MAE for numeric predictions. Report Accuracy and AUC for binary classifications. Report Accuracy for multi-class classifications.
from sklearn.metrics import mean_squared_error, mean_absolute_error

# MF evaluation
mf_preds = mf_model.predict((test_users_idx, test_items_idx), verbose=0)
mf_rmse = np.sqrt(mean_squared_error(test_ratings, mf_preds))
mf_mae = mean_absolute_error(test_ratings, mf_preds)

# NCF evaluation
ncf_preds = ncf_model.predict((test_users_idx, test_items_idx), verbose=0)
ncf_rmse = np.sqrt(mean_squared_error(test_ratings, ncf_preds))
ncf_mae = mean_absolute_error(test_ratings, ncf_preds)

print(f"\n{'Model':<10} {'RMSE':<10} {'MAE':<10}")
print(f"{'MF':<10} {mf_rmse:<10.4f} {mf_mae:<10.4f}")
print(f"{'NCF':<10} {ncf_rmse:<10.4f} {ncf_mae:<10.4f}")

Sample predictions:
user_idx     item_idx     actual     MF pred      NCF pred    
215326       59876        5.0        -0.0134      4.0049      
278875       57768        5.0        -0.0583      3.9556      
405797       34766        5.0        0.9598       3.9532      
56747        50084        1.0        0.3103       4.1764      
222374       61133        5.0        0.0008       4.0393      

Model      RMSE       MAE       
MF         3.9878     3.6647    
NCF        1.5905     1.2599    


#### **Interpretation:**

- MF's RMSE is 3.9878 and MAR is 3.6647, showing a high error which should be
  largely driven by the cold-start problem (96.8% of test instances involve unseen 
  users or items). MF relies on learned embedings from training, so it fails to generalize to new users and items, leads to weak performance for producing predictions.

- With deeper neural architecture capturing non-linear user-item relationships, NCF achived lower RMSE (1.5905) and MAE (1.2599) compared to MF. However, the RMSE and MAR are still relatively weak, we can also observe that predictions tend to cluster around 4.0 regardless of the true rating.

In [ ]:
# adding predictions to test_data
test_data = test_data.copy()
test_data['mf_pred'] = mf_preds
test_data['ncf_pred'] = ncf_preds

def evaluate_segment(df, label):
    mf_rmse = np.sqrt(mean_squared_error(df['rating'], df['mf_pred']))
    mf_mae = mean_absolute_error(df['rating'], df['mf_pred'])
    ncf_rmse = np.sqrt(mean_squared_error(df['rating'], df['ncf_pred']))
    ncf_mae = mean_absolute_error(df['rating'], df['ncf_pred'])
    print(f"\n[{label}] n={len(df):,}")
    print(f"  MF  → RMSE: {mf_rmse:.4f}, MAE: {mf_mae:.4f}")
    print(f"  NCF → RMSE: {ncf_rmse:.4f}, MAE: {ncf_mae:.4f}")
    return {'segment': label, 'n': len(df),
            'mf_rmse': mf_rmse, 'mf_mae': mf_mae,
            'ncf_rmse': ncf_rmse, 'ncf_mae': ncf_mae}

results = []

# Segment 1: User Activity
user_activity = train_data.groupby('user_id').size().reset_index(name='train_count')
test_data = test_data.merge(user_activity, on='user_id', how='left')
test_data['train_count'] = test_data['train_count'].fillna(0)

new_users_seg    = test_data[test_data['train_count'] == 0]
average_users_seg = test_data[(test_data['train_count'] >= 1) & (test_data['train_count'] <= 5)]
power_users_seg  = test_data[test_data['train_count'] > 5]

results.append(evaluate_segment(new_users_seg,     "New Users (0 interactions in train)"))
results.append(evaluate_segment(average_users_seg, "Average Users (1-5 interactions)"))
results.append(evaluate_segment(power_users_seg,   "Power Users (>5 interactions)"))

# Segment 2: Item Popularity
item_popularity = train_data.groupby('item_id').size().reset_index(name='item_train_count')
test_data = test_data.merge(item_popularity, on='item_id', how='left')
test_data['item_train_count'] = test_data['item_train_count'].fillna(0)

new_items_seg     = test_data[test_data['item_train_count'] == 0]
average_items_seg = test_data[(test_data['item_train_count'] >= 1) & (test_data['item_train_count'] <= 10)]
popular_items_seg = test_data[test_data['item_train_count'] > 10]

results.append(evaluate_segment(new_items_seg,     "New Items (0 interactions in train)"))
results.append(evaluate_segment(average_items_seg, "Average Items (1-10 interactions)"))
results.append(evaluate_segment(popular_items_seg, "Popular Items (>10 interactions)"))

# Segment 3: Cold-start vs Non-cold-start
cold_mask = (test_data['train_count'] == 0) | (test_data['item_train_count'] == 0)
cold_seg     = test_data[cold_mask]
non_cold_seg = test_data[~cold_mask]

results.append(evaluate_segment(cold_seg,     "Cold-Start Instances"))
results.append(evaluate_segment(non_cold_seg, "Non-Cold-Start Instances"))

# summary table
results_df = pd.DataFrame(results)
print("\n\nSummary Table:")
print(results_df[['segment', 'n', 'mf_rmse', 'ncf_rmse', 'mf_mae', 'ncf_mae']].to_string(index=False))


[New Users (0 interactions in train)] n=93,806
  MF  → RMSE: 3.9836, MAE: 3.6565
  NCF → RMSE: 1.5926, MAE: 1.2659

[Average Users (1-5 interactions)] n=5,937
  MF  → RMSE: 4.0464, MAE: 3.7787
  NCF → RMSE: 1.5095, MAE: 1.1385

[Power Users (>5 interactions)] n=257
  MF  → RMSE: 4.1677, MAE: 4.0185
  NCF → RMSE: 2.4331, MAE: 1.8653

[New Items (0 interactions in train)] n=41,978
  MF  → RMSE: 4.2531, MAE: 3.9876
  NCF → RMSE: 1.4892, MAE: 1.2342

[Average Items (1-10 interactions)] n=21,149
  MF  → RMSE: 4.0285, MAE: 3.7101
  NCF → RMSE: 1.6057, MAE: 1.2681

[Popular Items (>10 interactions)] n=36,873
  MF  → RMSE: 3.6368, MAE: 3.2711
  NCF → RMSE: 1.6903, MAE: 1.2845

[Cold-Start Instances] n=96,780
  MF  → RMSE: 3.9914, MAE: 3.6673
  NCF → RMSE: 1.5904, MAE: 1.2628

[Non-Cold-Start Instances] n=3,220
  MF  → RMSE: 3.8772, MAE: 3.5882
  NCF → RMSE: 1.5935, MAE: 1.1737


Summary Table:
                            segment     n  mf_rmse  ncf_rmse   mf_mae  ncf_mae
New Users (0 interact

#### **Deep Dive Interpretation**

**1.Segmentation Criteria & Justification** 

- User Activity: categorized by the number of interactions each user had in the
  training set. New Users (0 interactions, n=93,806) represent cold-start users
  the model has never seen. Average Users (1-5 interactions, n=5,937) reflect
  typical engagement levels. Power Users (>5 interactions, n=257) are the most
  active and loyal customers.

- Item Popularity: categorized by the number of interactions each item received
  in training. New Items (0 interactions, n=41,978) are freshly introduced titles
  with no history. Average Items (1-10 interactions, n=21,149) reflect moderate
  exposure. Popular Items (>10 interactions, n=36,873) are well-established titles
  with rich interaction history.

- Cold-Start vs Non-Cold-Start: defined by whether a test instance involves a new
  user or a new item. This directly measures the model's ability to handle
  real-world deployment scenarios where new users and items constantly appear.

---
**2.Performance by Segment**

[User Activity Segments]

NCF outperforms MF across all user segments, similar to previous evaluation.
For New Users, NCF achieves RMSE=1.59 vs MF's 3.98. Notably, NCF performs worst 
on Power Users (RMSE=2.43), possible explanation could be: Power Users have more complex and diverse preferences, yet there are only 257 such users in the test set, making it hard for model to learn their behavior. MF performs poorly across all user segments
(RMSE~4.0), confirming its fundamental inability to handle cold-start users.

[Item Popularity Segments]

MF shows its best performance on Popular Items (RMSE=3.64 vs 4.25 for New Items), as popular items have richer training histories for MF to learn from.
On the other hand, NCF shows the opposite trend: it performs best on New Items (RMSE=1.49) and slightly worse on Popular Items (RMSE=1.69). This suggests NCF has stronger
generalization capability and does not rely as heavily on item interaction history.

[Cold-Start vs Non-Cold-Start]

Both models show nearly identical performance on Cold-Start vs Non-Cold-Start instances. Since 96.8% of the test set are cold-start users, the overall metrics are already dominated by these interaction, leaving the non-cold-start segment (n=3,220) too small to show meaningful differentiation.

---
**3. Model Strengths and Weaknesses**

MF Strengths:
  - Simple, interpretable, and computationally efficient to train
  - Performs relatively better on popular items with rich interaction history

MF Weaknesses:
  - Completely fails on cold-start scenarios (RMSE~4.0 across all segments)
  - Predictions cluster near 0 rather than the valid rating range of 1-5,
    indicating the model has not learned meaningful user-item representations
  - Not suitable for production deployment given the 96.8% cold-start rate

NCF Strengths:
  - Substantially outperforms MF across all segments (RMSE improvement of ~60%)
  - Handles cold-start reasonably well due to its deeper neural architecture
  - Strong generalization on new items, making it more robust for a growing catalog

NCF Weaknesses:
  - Struggles with Power Users, suggesting difficulty capturing complex preferences
  - Tends to predict conservatively around 4.0 for most users, indicating a
    popularity bias that limits true personalization
  - Higher error on popular items suggests difficulty with polarized rating distributions
---
**4. Business & Product Implications**

The 96.8% cold-start rate reflects a rapidly growing platform where new users and
new game titles are continuously being added. While this is a positive trend showing that the platform consistently attract new comers, it is also a huge challenge for recommendation engine as it need to recommend new users with products it never seen before.

According to our evaluation, MF is not production-ready for this dataset and should only serve as an offline benchmark. NCF is a stronger candidate for deployment, but its popularity bias means it may over-recommend mainstream titles while failing to surface  games that really satisfy engaged users.

---
**5. Overall Recommendations**
- Incorporate side features such as game genre, platform (PS5/Xbox/PC), and price
  to address cold-start via content-based signals (e.g., Wide & Deep model)
- Adopt a hybrid approach combining collaborative filtering with content-based
  features specifically for new users and new items
- Collect richer behavioral data (e.g., playtime, wishlists, browsing history)
  beyond explicit ratings to build more robust user profiles
- Explore sequential models such as SASRec once sufficient interaction sequences
  are available, leveraging the temporal ordering already captured by timestamps